# 📖 Notebook 1: Cluster Setup — Your First Kubernetes Cluster

Welcome! In this notebook, you'll create a small local Kubernetes cluster with **minikube** and learn how to inspect it with **kubectl**.

Think of this notebook as your first guided tour of the Kubernetes control room.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Explain what Kubernetes does in simple words
- Name the main control plane and node components
- Verify that Docker, minikube, and kubectl are available
- Start a local Kubernetes cluster with minikube
- Explore nodes, namespaces, and system pods
- Use basic `kubectl` commands to inspect running workloads
- Run and delete your first pod

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['docker', 'minikube', 'kubectl']
INSTALL_HINTS = {
    'docker': 'https://docs.docker.com/get-docker/  (or `brew install --cask docker`)',
    'minikube': 'https://minikube.sigs.k8s.io/docs/start/  (or `brew install minikube`)',
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

print('Preflight OK:', ', '.join(REQUIRED))

## 🛠️ Setup

This lab assumes you're inside the `03-technologies/container-orchestration/kubernetes` project folder and using the notebooks from `notebooks/`.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

### Before You Start

Kubernetes will use Docker to run containers locally through minikube's Docker driver.
If Docker is not running, cluster creation will fail.

### Exercise

Run the commands below to make sure Docker responds and to see whether minikube is already running.

In [ ]:
!docker info --format 'Docker Server Version: {{.ServerVersion}}'
!minikube status

## 🤔 What is Kubernetes?

Imagine you manage a food court with many small kitchens.
Each kitchen is a **container**.
If one kitchen stops working, somebody needs to restart it.
If the lunch rush gets busy, somebody needs to open more kitchens.
If you want to move a kitchen to a different room, somebody has to keep track of that too.

**Kubernetes is the manager for all of those kitchens.**
It decides where containers should run, restarts them when they fail, and helps you scale them up when more people arrive.

In this lab, **minikube** creates a small local Kubernetes cluster and **kubectl** is the command-line remote control you use to talk to that cluster.

### Exercise

Check that your Kubernetes tools are installed.

In [ ]:
!minikube version
!kubectl version --client

## 🏗️ Cluster Architecture at a High Level

A Kubernetes cluster has two big sides:
- The **control plane**: the brains that decide what should happen
- The **node components**: the workers that actually run your containers

```text
                    ┌───────────────────────────────┐
                    │         Control Plane         │
                    │                               │
kubectl ───────────▶│  API Server                   │
                    │      │                        │
                    │      ├── Scheduler            │
                    │      ├── Controller Manager   │
                    │      └── etcd                 │
                    └──────────────┬────────────────┘
                                   │
                      desired state │ real state
                                   ▼
                    ┌───────────────────────────────┐
                    │           Worker Node         │
                    │                               │
                    │  kubelet                      │
                    │  kube-proxy                   │
                    │  container runtime            │
                    │  your Pods                    │
                    └───────────────────────────────┘
```

Quick mental model:
- **API Server** = front desk for all cluster requests
- **Scheduler** = chooses which node should run a pod
- **Controller Manager** = keeps trying to make reality match your YAML
- **etcd** = tiny database that stores cluster state
- **kubelet** = node agent that makes sure containers are running
- **kube-proxy** = helps networking and service routing

### Exercise

Create your local cluster. This may take a few minutes the first time.

### Sizing this cluster (read before you run the next cell)

Notebooks 5, 7 and 8 install Prometheus + Grafana, ArgoCD and Istio **into this same
cluster**. Those are real distributed systems, not toys, and together they want several
CPUs and a few GB of RAM. Start too small and you will not get an error message — you
will get pods stuck in `Pending` with `0/1 nodes are available: insufficient cpu`, which
is a much more confusing failure. So we size the cluster correctly now:

| Setting | Value | Why |
|---|---|---|
| `--cpus` | `4` | Istio's control plane + sidecars, Prometheus, ArgoCD |
| `--memory` | `6144` minimum, `8192` if you have it | kube-prometheus-stack asks for the better part of a gigabyte |
| `--driver` | `docker` | Docker Desktop must be running |

**minikube cannot be given more memory than Docker itself has.** Ask for 8192 on a Docker
Desktop configured with 8 GB and minikube refuses to start — the VM has to fit *inside*
Docker's allocation, with room left for the daemon. The next cell therefore reads Docker's
own limit and picks the largest safe size instead of hard-coding one, so it works whether
your Docker Desktop is set to 8 GB or to 16. If it lands on the 6144 floor, raise Docker
Desktop's memory in Settings → Resources and re-run.

**One more thing worth knowing now**: minikube's default networking plugin (CNI) does
**not enforce NetworkPolicy**. NetworkPolicy objects will be created and will look fine
in `kubectl get netpol`, but nothing blocks traffic. Notebook 6 depends on enforcement,
so if you want the security demos there to actually block anything, start the cluster
with a policy-capable CNI instead:

```bash
minikube start --cpus=4 --memory=6144 --driver=docker --cni=calico
```

Calico costs a little extra CPU and RAM. Notebook 6 detects which one you chose and
tells you what to expect either way, so it is safe to start without it and come back.

In [ ]:
# Later notebooks install Prometheus, ArgoCD and Istio into this same cluster.
# 2 CPU / 4 GB is not enough for those -- pods end up Pending on insufficient
# cpu/memory. Size the cluster once, here, and you never have to redo it.
#
# The size is computed rather than hard-coded because `minikube start --memory`
# is a HARD failure when it exceeds what Docker itself has: minikube's node is a
# container inside Docker's allocation, so asking for 8192 on an 8 GB Docker
# Desktop fails with "Requested memory allocation is more than your system limit".
import json
import subprocess

FLOOR_MB, CEILING_MB, DAEMON_HEADROOM_MB = 6144, 8192, 1536

info = subprocess.run(["docker", "info", "--format", "{{json .}}"],
                      capture_output=True, text=True)
if info.returncode != 0:
    raise RuntimeError("Docker is not responding. Start Docker Desktop and re-run.\n"
                       + info.stderr.strip()[:300])

docker_mb = json.loads(info.stdout)["MemTotal"] // (1024 * 1024)
usable = docker_mb - DAEMON_HEADROOM_MB
MEMORY_MB = max(FLOOR_MB, min(CEILING_MB, (usable // 256) * 256))

print(f"Docker Desktop has {docker_mb} MiB; starting minikube with {MEMORY_MB} MiB.")
if usable < FLOOR_MB:
    print(f"⚠️  That is below the {FLOOR_MB} MiB floor this series needs. minikube may")
    print("   refuse to start, and notebooks 5/7/8 will produce Pending pods if it does.")
    print("   Raise Docker Desktop -> Settings -> Resources -> Memory to at least 8 GB.")

!minikube start --cpus=4 --memory={MEMORY_MB} --driver=docker

# Already running with a smaller profile? Recreate it:
#   minikube delete && minikube start --cpus=4 --memory=6144 --driver=docker

## 🔎 Explore the Cluster

Once the cluster is running, start by asking three beginner-friendly questions:
1. **Where is the cluster control plane?**
2. **What nodes exist?**
3. **What namespaces are already there?**

A **namespace** is a logical folder inside Kubernetes. It helps separate different groups of resources.

### Exercise

Run the commands below and read the output slowly.

In [ ]:
!kubectl cluster-info
!kubectl get nodes
!kubectl get namespaces

In [ ]:
# A notebook that only prints is a notebook that can fail quietly. From here on,
# every section that claims an outcome also checks it -- so a broken cluster stops
# the notebook here instead of producing ten cells of confusing output later.
import json
import subprocess


def kget(*args):
    """kubectl get ... -o json, parsed. Raises with kubectl's own message on failure."""
    r = subprocess.run(["kubectl", "get", *args, "-o", "json"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


nodes = kget("nodes")["items"]
assert len(nodes) == 1, f"expected a single-node minikube cluster, got {len(nodes)}"

node = nodes[0]
ready = [c for c in node["status"]["conditions"] if c["type"] == "Ready"]
assert ready and ready[0]["status"] == "True", \
    f"node {node['metadata']['name']} is not Ready: {ready}"

alloc = node["status"]["allocatable"]
cpu = int(alloc["cpu"])
mem_mb = int(alloc["memory"].rstrip("Ki")) // 1024
assert cpu >= 4, f"cluster has only {cpu} allocatable CPUs; notebooks 5/7/8 need 4"
assert mem_mb >= 5000, f"cluster has only {mem_mb} MiB allocatable; need ~6 GB"

print(f"✅ node {node['metadata']['name']}: Ready, {cpu} cpu, {mem_mb} MiB allocatable")

# One gotcha worth knowing before you fill this cluster up. With the docker
# driver, the node is a container with a hard memory limit -- but on macOS and
# Windows the kubelet reads the HOST's memory, not the container's cgroup limit,
# and reports that as the node's capacity. The scheduler then believes it has
# more room than Docker will actually allow.
#
# The consequence is not a normal eviction. The kubelet never sees memory
# pressure, so it never evicts anything; Docker kills or stalls the whole node
# container instead, and what you observe is the entire cluster going away
# (`Unable to connect to the server: net/http: TLS handshake timeout`) rather
# than one pod dying. If that happens later in the series, this is why.
if mem_mb > MEMORY_MB + 256:
    print()
    print(f"⚠️  The node reports {mem_mb} MiB but minikube was given {MEMORY_MB} MiB.")
    print("   The kubelet is reading host memory, not its own cgroup limit, so it")
    print("   will happily schedule past the real ceiling. Keep an eye on")
    print(f"   `docker stats minikube` -- the limit that matters is {MEMORY_MB} MiB.")

## ⚙️ Explore System Components

When minikube starts a cluster, Kubernetes launches several system pods in a namespace called `kube-system`.
These pods are the internal plumbing that keeps the cluster alive.

You do **not** need to memorize every pod name yet.
At this stage, the goal is simply to notice that Kubernetes is already running lots of supporting pieces before you deploy your own app.

### Exercise

List the internal system pods.

In [ ]:
!kubectl get pods -n kube-system

## ⌨️ kubectl Basics: get, describe, and logs

`kubectl` has a few commands you'll use constantly:
- `kubectl get` gives you a list view
- `kubectl describe` gives you a detailed story about one resource
- `kubectl logs` shows output from a container

Right now, let's use `get` and `describe` on things that already exist.
We'll use `logs` right after we start our first pod.

### Exercise

Inspect the cluster using both the short view and the detailed view.

In [ ]:
!kubectl get pods -A
!kubectl describe node minikube

## 🚀 Run Your First Pod

A **pod** is the smallest thing Kubernetes schedules.
You can think of it as a wrapper around one or more containers that should live together.

Let's create a simple `nginx` pod. Then we'll use all three basic inspection commands:
- `get` to confirm it exists
- `describe` to inspect its details and events
- `logs` to read its container output

### Exercise

Create the pod and inspect it.

In [ ]:
# `kubectl run` fails with AlreadyExists on a second run, which makes the notebook
# not re-runnable. Rendering the pod with --dry-run=client and piping it into
# `kubectl apply` is the idempotent form -- run it as many times as you like.
!kubectl run nginx --image=nginx:1.27 --dry-run=client -o yaml | kubectl apply -f -
!kubectl wait --for=condition=Ready pod/nginx --timeout=120s
!kubectl get pods
!kubectl describe pod nginx
!kubectl logs nginx

# `kubectl wait` above exits non-zero on timeout, but a non-zero `!` command does
# NOT fail a notebook cell -- it just prints. Check the phase in Python so a pod
# that never came up stops the notebook instead of being scrolled past.
pod = kget("pod", "nginx")
assert pod["status"]["phase"] == "Running", \
    f"nginx pod is {pod['status']['phase']}, not Running -- read the Events above"
print("\n✅ nginx pod is Running")

### Where failures actually show up

Scroll to the bottom of the `kubectl describe pod nginx` output. There is an **Events**
section:

```text
Events:
  Type    Reason     Age   From               Message
  ----    ------     ----  ----               -------
  Normal  Scheduled  10s   default-scheduler  Successfully assigned default/nginx to minikube
  Normal  Pulling    10s   kubelet            Pulling image "nginx:1.27"
  Normal  Pulled     3s    kubelet            Successfully pulled image "nginx:1.27"
  Normal  Created    3s    kubelet            Created container nginx
  Normal  Started    3s    kubelet            Started container nginx
```

Almost every "why is my pod not running?" answer is in that list. Learn to read it now,
because the three most common Kubernetes failures all announce themselves there and
nowhere else:

| What you see in `kubectl get pods` | What Events says | What it means |
|---|---|---|
| `Pending` | `0/1 nodes are available: insufficient cpu` | The scheduler cannot fit the pod. Nothing is broken — the cluster is just full. |
| `ImagePullBackOff` | `Failed to pull image ...: not found` | Wrong image name/tag, or the image was never built into minikube. |
| `CrashLoopBackOff` | `Back-off restarting failed container` | The container starts and immediately exits. `kubectl logs --previous` has the reason. |

`kubectl get pods` tells you *that* something is wrong. `kubectl describe` tells you *why*.
`kubectl logs` tells you what the app itself said.

## 🧹 Clean Up

Cleaning up is part of working with Kubernetes.
If you leave resources around, future exercises can get confusing because old objects are still running.

### Exercise

Delete the `nginx` pod you created.

In [ ]:
# --ignore-not-found means this cell is safe to run twice, or to run when the
# pod was never created. Get in the habit: cleanup cells should never fail.
!kubectl delete pod nginx --ignore-not-found

## 🎓 What You Learned

In this notebook, you:
- Used Docker, minikube, and kubectl together
- Started a local Kubernetes cluster
- Saw the difference between control plane components and node components
- Explored nodes, namespaces, and `kube-system` pods
- Practiced `kubectl get`, `kubectl describe`, and `kubectl logs`
- Ran and deleted your first pod

You're ready for the next step: working with **Pods** and **Deployments** so Kubernetes can manage application replicas for you.

## ➡️ What Notebook 2 Needs From This One

These notebooks build on each other. From here on, each notebook opens by checking that
the previous one left the cluster in the right state:

- **Notebook 2** needs this running minikube cluster. It builds the three sample service
  images into it and deploys them into a namespace called `k8s-lab`.
- **Notebooks 3 to 10** all need that `k8s-lab` namespace and those three deployments.

So: do not `minikube delete` between notebooks. `minikube stop` / `minikube start` is
fine — it preserves everything.